Codagh
======

### Import

In [1]:
import os
import numpy as np
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import chess
from chess import pgn
from tqdm import tqdm

## Data preprocessing

### Loading data

In [ ]:
def get_number_of_games(file_path):
    number_of_games = 0
    with open(file_path, 'r') as pgn_file:
        while True:
            if not pgn.skip_game(pgn_file):
                break
            number_of_games += 1
    return number_of_games
    

def load_pgn(file_path, offset):
    games = np.memmap(filename="../lib/data/npy/games.npy", dtype='object', mode="r+")
    with open(file_path, 'r') as pgn_file:
        i = offset
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games[i] = game
            i += 1
    del games
    return i

files = [file for file in os.listdir("../lib/data/pgn") if file.endswith(".pgn")]
LIMIT_OF_FILES = min(len(files), 30)
number_of_games = 0
for file in tqdm(files[:LIMIT_OF_FILES]):
    number_of_games += get_number_of_games(f"../lib/data/pgn/{file}")

games = np.memmap(filename="../lib/data/npy/games.npy", dtype='object', mode="w+", shape=(number_of_games))
del games
offset = 0
for file in tqdm(files[:LIMIT_OF_FILES]):
    offset = load_pgn(f"../lib/data/pgn/{file}", offset)
games = np.memmap(filename="../lib/data/npy/games.npy", dtype='object', mode="r+")

 97%|█████████▋| 29/30 [02:56<00:20, 20.10s/it]

In [ ]:
print(f"Games parsed: {len(games)}")

### Convert data into tensors

In [ ]:
from ridoc import generate_nn_input

In [ ]:
positions, moves = generate_nn_input(games)
print(f"Number of samples: {len(moves)}")

## Preliminary actions

In [ ]:
from jesinia import ChessDataset
from violet import ChessModel

In [ ]:
dataset = ChessDataset(positions, moves)

dataloader = DataLoader(dataset, batch_size=64)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = ChessModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Traning

In [ ]:
num_epochs = 30
for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    running_loss = 0.0
    for inputs, lables in tqdm(dataloader):
        inputs, lables = inputs.to(device), lables.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, lables)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        running_loss += loss.item()
    end_time = time.time()
    epoch_time = end_time - start_time
    minutes: int = int(epoch_time // 60)
    seconds: int = int(epoch_time) - minutes * 60
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {running_loss / len(dataloader):.4f}, Time: {minutes}m{seconds}s")

### Save the model

In [ ]:
model_name = f"2_2_{LIMIT_OF_FILES}_{num_epochs}"
torch.save(model.state_dict(), f"../models/{model_name}.pth")